## Módulo 1 - Sistema Baseado em Conhecimento para Gestão de Emergências

### 1. Introdução
**Contexto**

A Proteção Civil da cidade pretende um sistema de apoio à decisão para avaliar alertas ambientais em tempo real. Por exemplo, detetarpicos de calor, risco de incêndio, precipitação alta e vento forte.

**Objetivos de Aprendizagem**
1.	Reconhecer como a lógica de predicados e as regras “se-então” estruturam conhecimento;
2.	Aplicar raciocínio dedutivo e heurísticas para priorizar respostas;
3.	Compreender incerteza usando uma Rede Bayesiana simples.

**Tarefas**
1.	Definir pelo menos 10 regras de risco em lógica de predicados (“Se temperatura > 40 °C ∧ humidade < 20 % → risco_incêndio_alto”);
2.	Implementar um motor de inferência que, tendo por base um knowledge base, leia o dataset e indique ações recomendadas para cada caso presente no ficheiro.
3.	Criar uma Rede Bayesiana com 3-4 nós e usar inferência por enumeração para atualizar as probabilidades.



### 2. Configuração do Ambiente e Importação de Dados
    2.1. Importação de Bibliotecas

In [7]:
import sys
print(sys.executable) 
import pandas as pd
print(pd.__version__)
import json
from pathlib import Path

c:\ProgramData\anaconda3\python.exe
2.0.3


    2.3. Carregamento do Dataset

In [21]:
INPUT_CSV = '../data/processed_lisboa_porto_air_quality.csv'
RULES_JSON="regras.json"
RULES_OUTPUT_CSV = 'resultados_alertas.csv'

df = pd.read_csv(INPUT_CSV, sep=';')
print("Dataset carregado")

# Visualizar as primeiras linhas +. nomes das colunas
display(df.head())
print("\nColunas disponíveis no dataset:")
print(df.columns.tolist())

Dataset carregado


,city,datetime,CO,NO2,O3,PM10,PM2.5,SO2,temperature_c,humidity_percent,pressure_hpa,wind_speed_kmh,wind_direction_deg,precipitation_mm,C6H6,NMHC,NOx,air_quality_good,year,month
0,Lisboa,05/09/25 01:00,0.96,25.20,84.09,11.82,9.12,6.75,18.9,82.0,1018.5,16.2,357.0,0.0,NaN,NaN,NaN,True,2025,9
1,Lisboa,05/09/25 02:00,0.75,26.40,86.20,13.24,8.87,5.11,18.8,80.0,1018.3,15.5,356.0,0.0,NaN,NaN,NaN,True,2025,9
2,Lisboa,05/09/25 03:00,0.87,25.16,74.41,15.18,10.84,5.76,18.6,79.0,1017.4,11.5,2.0,0.0,NaN,NaN,NaN,True,2025,9
3,Lisboa,05/09/25 04:00,0.51,13.59,68.57,17.48,13.14,5.03,18.3,77.0,1017.2,11.3,351.0,0.0,NaN,NaN,NaN,True,2025,9
4,Lisboa,05/09/25 05:00,0.61,15.89,78.79,14.70,13.68,6.20,18.6,72.0,1016.8,9.4,356.0,0.0,NaN,NaN,NaN,True,2025,9



Colunas disponíveis no dataset:
['city', 'datetime', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2', 'temperature_c', 'humidity_percent', 'pressure_hpa', 'wind_speed_kmh', 'wind_direction_deg', 'precipitation_mm', 'C6H6', 'NMHC', 'NOx', 'air_quality_good', 'year', 'month']


    2.3. Tratamento dos dados

In [9]:
# Ver tipos de dados (features em colunas)
print("Tipos de dados:")
df.dtypes.to_frame("dtype")

Tipos de dados:


,dtype
city,object
datetime,object
CO,float64
NO2,float64
O3,float64
PM10,float64
PM2.5,float64
SO2,float64
temperature_c,float64
humidity_percent,float64


In [10]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
CO,9116.0,1.971439,1.408473,0.10,0.9400,1.600,2.6000,11.90
NO2,9157.0,99.959433,54.052344,2.00,57.1200,98.000,134.0000,340.00
O3,1442.0,69.301706,16.780429,28.22,56.5525,69.205,81.3100,114.68
PM10,1442.0,21.614896,9.428184,1.84,13.9525,20.110,28.0200,58.57
PM2.5,1442.0,15.138800,6.889596,0.10,9.6500,14.080,19.9400,38.75
SO2,1442.0,6.453259,3.050010,0.10,4.1225,6.010,8.3975,18.36
temperature_c,10433.0,18.471341,8.327228,-1.90,12.7000,18.200,23.8000,44.60
humidity_percent,10433.0,51.978213,18.949539,9.20,37.6000,51.700,65.6000,100.00
pressure_hpa,1442.0,1019.587864,3.062214,1002.90,1018.2000,1020.100,1021.6000,1025.70
wind_speed_kmh,1442.0,8.953051,6.143344,0.00,3.7000,7.300,13.0000,28.00


In [11]:
# Contar duplicados
print(f"Duplicados: {df.duplicated().sum()}")

Duplicados: 0


In [12]:
# Resumo de qualidade (missing + cardinalidade), já ordenado por missing
summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True),
    "unique_pct": ((df.nunique(dropna=True) / len(df)) * 100).round(2),
}).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

print("Resumo de missing e cardinalidade (ordenado):")
display(summary)

Resumo de missing e cardinalidade (ordenado):


,missing_count,missing_pct,n_unique,unique_pct
NMHC,9854,91.51,429,3.98
O3,9326,86.61,1281,11.90
PM10,9326,86.61,1179,10.95
PM2.5,9326,86.61,1119,10.39
SO2,9326,86.61,807,7.49
wind_direction_deg,9326,86.61,299,2.78
wind_speed_kmh,9326,86.61,242,2.25
pressure_hpa,9326,86.61,154,1.43
precipitation_mm,9326,86.61,18,0.17
NOx,3050,28.32,925,8.59


In [13]:
# Resumo de qualidade (missing + cardinalidade), já ordenado por missing - Só para Lisboa e Porto
df_filtrado = df[df['city'].isin(['Lisboa', 'Porto'])]
summary = pd.DataFrame({
    "missing_count": df_filtrado.isna().sum(),
    "missing_pct": (df_filtrado.isna().mean() * 100).round(2),
    "n_unique": df_filtrado.nunique(dropna=True),
    "unique_pct": ((df_filtrado.nunique(dropna=True) / len(df_filtrado)) * 100).round(2),
}).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

print("Resumo de missing e cardinalidade (ordenado) para Lisboa e Porto:")
display(summary)

Resumo de missing e cardinalidade (ordenado) para Lisboa e Porto:


,missing_count,missing_pct,n_unique,unique_pct
C6H6,1442,100.0,0,0.00
NMHC,1442,100.0,0,0.00
NOx,1442,100.0,0,0.00
O3,0,0.0,1281,88.83
NO2,0,0.0,1247,86.48
PM10,0,0.0,1179,81.76
PM2.5,0,0.0,1119,77.60
SO2,0,0.0,807,55.96
datetime,0,0.0,721,50.00
wind_direction_deg,0,0.0,299,20.74


In [14]:
#Eliminar as colunas que não são necessárias para a análise (C6H6, NMHC, NOx)
colunas_para_eliminar = ['C6H6', 'NMHC', 'NOx']
df = df.drop(columns=colunas_para_eliminar)
print(f"Colunas eliminadas: {colunas_para_eliminar}")

Colunas eliminadas: ['C6H6', 'NMHC', 'NOx']


In [15]:
# Confirmar que as colunas foram eliminadas
df.head()

,city,datetime,CO,NO2,O3,PM10,PM2.5,SO2,temperature_c,humidity_percent,pressure_hpa,wind_speed_kmh,wind_direction_deg,precipitation_mm,air_quality_good,year,month
0,Lisboa,05/09/25 01:00,0.96,25.20,84.09,11.82,9.12,6.75,18.9,82.0,1018.5,16.2,357.0,0.0,True,2025,9
1,Lisboa,05/09/25 02:00,0.75,26.40,86.20,13.24,8.87,5.11,18.8,80.0,1018.3,15.5,356.0,0.0,True,2025,9
2,Lisboa,05/09/25 03:00,0.87,25.16,74.41,15.18,10.84,5.76,18.6,79.0,1017.4,11.5,2.0,0.0,True,2025,9
3,Lisboa,05/09/25 04:00,0.51,13.59,68.57,17.48,13.14,5.03,18.3,77.0,1017.2,11.3,351.0,0.0,True,2025,9
4,Lisboa,05/09/25 05:00,0.61,15.89,78.79,14.70,13.68,6.20,18.6,72.0,1016.8,9.4,356.0,0.0,True,2025,9


### 3. Definição da Base de Conhecimento (regras.json)

In [16]:
rules_path = Path(RULES_JSON)
if not rules_path.exists():
    raise FileNotFoundError("Falta regras.json em Module_1")

with rules_path.open("r", encoding="utf-8") as f:
    rules_payload = json.load(f)

rules = rules_payload.get("rules", []) if isinstance(rules_payload, dict) else rules_payload
print(f"Total de regras: {len(rules)}")
display(pd.DataFrame([{
    "id": r.get("id"),
    "description": r.get("description"),
    "priority": r.get("priority"),
    "risk_level": (r.get("consequence") or {}).get("risk_level", r.get("risk_level"))
} for r in rules]))

Total de regras: 12


,id,description,priority,risk_level
0,R01_NO2_ALTO,Alerta NO2 critico - limite horario UE,10,ALTO
1,R02_NO2_MODERADO,Alerta NO2 preventivo,5,MODERADO
2,R03_PM10_ALTO,Particulas inalaveis PM10 excedem limite 24h,9,ALTO
3,R04_PM25_ALTO,Particulas finas PM2.5 excedem limite anual UE,10,ALTO
4,R05_O3_ALTO,Ozono troposferico - limiar de informacao,8,ALTO
5,R06_CO_ALTO,Monoxido de carbono - media 8h critica,9,ALTO
6,R07_SO2_ALTO,Dioxido de enxofre - limite horario,7,ALTO
7,R08_CALOR_EXTREMO,Onda de calor extremo,10,ALTO
8,R09_RISCO_INCENDIO,Risco maximo de incendio florestal,10,ALTO
9,R10_VENTO_FORTE,Vento forte - alerta laranja,7,ALTO


### 4. Implementação do Motor de Inferência (rules_engine.py)

In [ ]:
from rules_engine import (run_inference, mostrar_relatorio)

try:
    rules_output = pd.read_csv(RULES_OUTPUT_CSV, sep=';')
except:
    rules_output = run_inference(csv_path=INPUT_CSV, rules_path=RULES_JSON)

mostrar_relatorio(rules_output)



RESUMO DOS ALERTAS GERADOS
Total de observações : 10768
Com alerta           : 4661 (43.3%)
Sem alerta           : 6107 (56.7%)

Por nível de risco:
  NORMAL       6107  (56.7%)
  MODERADO     3828  (35.5%)
  ALTO          833  (7.7%)

Regras mais disparadas (Top 5):
  R02_NO2_MODERADO                4059 vezes
  R01_NO2_ALTO                    398 vezes
  R09_RISCO_INCENDIO              388 vezes
  R12_QUALIDADE_AR_PESSIMA        66 vezes
  R08_CALOR_EXTREMO               61 vezes

Por cidade:
  Lisboa           13/721 observações com alerta
  Porto            10/721 observações com alerta
  UCI_Dataset      4638/9326 observações com alerta


### 5. Execução e Análise de Resultados

### 6. Modelagem de Incerteza: Rede Bayesiana (bayes_alerts.py)

### 7. Inferência por Enumeração

### 8. Conclusão Crítica e Ética